Relic of Lyhr Analysis

**Anima Evasen**
- 6:00 AM
 
I figured out the pseudo code, I can help program the lyhr origin damage

When strike damage is received if received strike damage is lyhr && wearer contains lyhr then add damage to lyhrDamageTotal


**Anima Evasen**
- 6:27 AM

|Relic Wearer's Damage |Ally's Final Incoming Damage x 50%|
|:----|----:|
|5657 * 50% = 2828.5  | Zell's Damage Taken * 50% |
|6534 * 50% = 3267    | Zell's Total Damage Taken * 50%|

|Personal Damage Reduction |Toughness/Base Armor (Light)|
|:----|----:|
|3211/4252 = 0.75|      // 4 Instrument = 75%  Damage Reduction|
|3077/4118 = 0.74|      // 3 Instrument|
|2944/3985 = 0.73|   // 2 Instrument|
|2810/3851 = 0.729|  // 1 Instrument|

If full stars sigil stack, no guild guild objective buff, superior borderlands bloodlust, presence to the keep

__Final Damage = Lyhr Strike Damage * (1-Damage Reduction)__

|__Damage Taken__|
|:----|
|2828.5 * (1-75%) = 707 |
|2828.5 * (1-74%) = 735 |
|2828.5 * (1-73%) = 763|
|2828.5 * (1-72.9%) = 766.52|

|__Total Damage Taken__|
|----|
|3267 * (1-75%)  | 816|
|3267 * (1-74%) | 849|
|3267 * (1-73%) | 882|
|3267 * (1-72.9%) | 885|


In [34]:
from dataclasses import dataclass,field
import os.path
from os import listdir
import sys
from enum import Enum
import importlib
import xlrd
from xlutils.copy import copy
import json
import jsons
import math
import requests
import datetime
import gzip

from collections import OrderedDict

#Change input_directory to Elite Insight log directory
input_directory = 'c:\\GW2Logs\\Output\\test\\'
files = listdir(input_directory)
sorted_files = sorted(files)


#Sound remain constant
Lyhr_ID = 70353
Lyhr_Buff_Active = {}
Lyhr_Buff_Damage = {}

#loop through all files
for filename in sorted_files:
    # skip files of incorrect filetype
    file_start, file_extension = os.path.splitext(filename)
    #if args.filetype not in file_extension or "top_stats" in file_start:
    if file_extension not in ['.json', '.gz']:
        continue
        
    file_path = "".join((input_directory,"/",filename))

    if file_extension == '.gz':
        with gzip.open(file_path, mode="r") as f:
            json_data = json.loads(f.read().decode('utf-8'))
    else:
        json_datafile = open(file_path, encoding='utf-8')
        json_data = json.load(json_datafile)    

        const_fight = {}
        fight_data = json_data['players'] 
        output = []
        header = ["Fights"]

        for player in fight_data:
            name = player['name']
            lyhr_buff_check = False
            for buff in player['buffUptimes']:
                if buff['id'] != Lyhr_ID:
                    continue
                else:
                    lyhr_buff_check = True                   
                    if name not in Lyhr_Buff_Active:
                        Lyhr_Buff_Active[name] = {}
                    for buffer in buff['statesPerSource']:
                        if buffer not in Lyhr_Buff_Active[name]:
                            Lyhr_Buff_Active[name][buffer]={}
                        for state in buff['statesPerSource'][buffer]:
                            Lyhr_Buff_Active[name][buffer][state[0]]=state[1]
                    print(f"Player: {player['name']} needs Lyhr review in fight {file_start}")
            if lyhr_buff_check:
                #get player powerDamageTaken1S
                if name not in Lyhr_Buff_Damage:
                    Lyhr_Buff_Damage[name]={}
                for i in range(len(player['powerDamageTaken1S'][0])):
                    if i == 0:
                        damage_since_last = player['powerDamageTaken1S'][0][i]
                    else:
                        damage_since_last = player['powerDamageTaken1S'][0][i] - player['powerDamageTaken1S'][0][i-1]
                    Lyhr_Buff_Damage[name][i]=damage_since_last
                
                



Player: Buffalo Wyld Wings needs Lyhr review in fight 20251111-182815_detailed_wvw_kill
Player: Esonavi needs Lyhr review in fight 20251111-182815_detailed_wvw_kill
Player: Melody Rin needs Lyhr review in fight 20251111-182815_detailed_wvw_kill
Player: Pansofia Athanasios needs Lyhr review in fight 20251111-182815_detailed_wvw_kill
Player: Præsto Sum needs Lyhr review in fight 20251111-182815_detailed_wvw_kill


In [24]:
print("--Relic of Lyhr Buff--")
for player in Lyhr_Buff_Active:
    print(f"Player: {player}")
    for buffer in Lyhr_Buff_Active[player]:
        print(f"\tBuff Provider:\t{buffer}")
        print(f"\t\tTime\tStatus")
        for timeMS in Lyhr_Buff_Active[player][buffer]:
            fight_time = timeMS/1000
            status = bool(Lyhr_Buff_Active[player][buffer][timeMS])
            print(f"\t\t{fight_time}\t{status}")

--Relic of Lyhr Buff--
Player: Buffalo Wyld Wings
	Buff Provider:	Anima Evasen
		Time	Status
		0.0	False
		1.676	True
		6.676	False
Player: Esonavi
	Buff Provider:	Anima Evasen
		Time	Status
		0.0	False
		1.676	True
		6.676	False
		20.448	True
		25.448	False
Player: Melody Rin
	Buff Provider:	Anima Evasen
		Time	Status
		0.0	False
		1.676	True
		6.676	False
		20.448	True
		24.737	False
Player: Pansofia Athanasios
	Buff Provider:	Anima Evasen
		Time	Status
		0.0	False
		20.448	True
		25.448	False
Player: Præsto Sum
	Buff Provider:	Anima Evasen
		Time	Status
		0.0	False
		1.676	True
		6.676	False
		20.448	True
		25.448	False


In [46]:
from typing import Dict

def damage_while_buff_active(
    damage: Dict[int, int],
    buff: Dict[str, Dict[int, int]],
    buff_name: str = "Anima Evasen"
) -> int:
    """
    Return total damage taken *while* the named buff is active.

    Parameters
    ----------
    damage : dict
        {second: damage, ...}
    buff : dict
        {buff_name: {ms: 0/1, ...}}
    buff_name : str
        Buff to check

    Returns
    -------
    int
        Sum of damage for every second that contains at least one
        millisecond where the buff was active.
    """
    timeline = buff.get(buff_name, {})
    if not timeline:
        return 0

    events = sorted(timeline.items())
    total = 0
    i = 0
    n = len(events)

    while i < n:
        start_ms, state = events[i]

        end_ms = None
        j = i + 1
        while j < n:
            nxt_ms, nxt_state = events[j]
            if nxt_state != state:
                end_ms = nxt_ms
                break
            j += 1
        else:
            end_ms = None

        
        # If the buff is ON, sum damage in the interval
        if state in (1, True):
            cur_ms = start_ms
            last_sec = -1
            while True:
                sec = cur_ms // 1000

                # add damage only the first time we see a new second
                if sec != last_sec:
                    total += damage.get(sec, 0)
                    last_sec = sec

                # stop condition
                if end_ms is not None and cur_ms >= end_ms:
                    break
                if end_ms is None and cur_ms > start_ms + 1_000_000:  # safety
                    break
                cur_ms += 1

        # advance to next segment
        i = j if end_ms is not None else n

    return total

In [51]:
print("Power Damage Taken while Relic of Lyhr buff was active")
for player in Lyhr_Buff_Active:
    damage=Lyhr_Buff_Damage[player]
    buff=Lyhr_Buff_Active[player]
    lyhr_buffers = []
    for buffer in Lyhr_Buff_Active['Buffalo Wyld Wings']:
        lyhr_buffers.append(buffer)
    print(player)
    print(f"\t{damage_while_buff_active(damage, buff)}\t{lyhr_buffers}")

Power Damage Taken while Relic of Lyhr buff was active
Buffalo Wyld Wings
	3084	['Anima Evasen']
Esonavi
	13239	['Anima Evasen']
Melody Rin
	4710	['Anima Evasen']
Pansofia Athanasios
	6345	['Anima Evasen']
Præsto Sum
	21119	['Anima Evasen']
